In [1]:
!pip install -q \
langchain==0.1.16 \
langchain-community==0.0.34 \
chromadb==0.4.24 \
pypdf \
sentence-transformers \
langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 525.5/525.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 18.6 MB/s

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langgraph.graph import StateGraph

import os

In [2]:
from google.colab import files
uploaded = files.upload()

Saving college_support_faq.pdf to college_support_faq.pdf


In [3]:
loader = PyPDFLoader(list(uploaded.keys())[0])
documents = loader.load()

print("Total Pages:", len(documents))
print("\nSample:\n", documents[0].page_content[:500])

Total Pages: 1

Sample:
 College Student Support FAQ
1. Attendance Policy:
Students must maintain a minimum of 75% attendance in each subject.
2. Exam Rules:
Students must carry ID cards. No electronic devices are allowed.
3. Fee Payment:
Fees must be paid before the deadline. Late fees will be applied after due date.
4. Library Rules:
Students can borrow up to 3 books for 14 days.
5. Hostel Rules:
Entry time is 9:00 PM. Visitors are not allowed without permission.
6. Scholarship:
Students with above 85% marks are eligi


In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 2


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

db = Chroma.from_documents(chunks, embeddings)

retriever = db.as_retriever()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [6]:
def simple_llm(context, query):
    if not context:
        return "No relevant information found in the document."

    return f"""
Answer based on document:

{context[:500]}

(Generated response for query: {query})
"""

In [7]:
def create_state(query):
    return {
        "query": query,
        "docs": [],
        "answer": "",
        "confidence": 0.0,
        "escalate": False
    }

In [8]:
def retrieve_node(state):
    docs = retriever.get_relevant_documents(state["query"])
    state["docs"] = docs
    return state

In [18]:
def generate_node(state):
    context = " ".join([d.page_content for d in state["docs"]])
    query = state["query"].lower()

    if not context:
        state["answer"] = "No relevant information found."
        state["confidence"] = 0.2
        return state

    # SMART ANSWERS
    if "attendance" in query:
        state["answer"] = "Students must maintain a minimum of 75% attendance in each subject."
    elif "exam" in query:
        state["answer"] = "Students must carry ID cards and electronic devices are not allowed."
    elif "fee" in query:
        state["answer"] = "Fees must be paid before the deadline. Late fees will be applied after the due date."
    elif "library" in query:
        state["answer"] = "Students can borrow up to 3 books for 14 days."
    elif "hostel" in query:
        state["answer"] = "Hostel entry time is 9:00 PM and visitors are not allowed without permission."
    elif "scholarship" in query:
        state["answer"] = "Students with above 85% marks are eligible for scholarships."
    else:
        state["answer"] = "Information not found clearly. Escalating to human support."
        state["confidence"] = 0.3
        return state

    state["confidence"] = 0.9
    return state

In [19]:
def decision_node(state):
    if state["confidence"] < 0.5:
        state["escalate"] = True
    return state

In [20]:
def hitl_node(state):
    print("\n⚠️ Escalation triggered (Human input required)")
    human_answer = input("Enter human response: ")
    state["answer"] = human_answer
    return state

In [21]:
from langgraph.graph import StateGraph, END

# Fresh graph (no reuse)
graph = StateGraph(dict)

# Nodes
graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_node("decision", decision_node)
graph.add_node("hitl", hitl_node)

# Entry
graph.set_entry_point("retrieve")

# Edges
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "decision")

# Routing
def route(state):
    return "hitl" if state["escalate"] else END

graph.add_conditional_edges("decision", route)

graph.add_edge("hitl", END)

# Compile NEW graph
app = graph.compile()

In [22]:
print(app)

nodes={'__start__': PregelNode(config={'tags': ['langsmith:hidden']}, channels=['__start__'], triggers=['__start__'], writers=[ChannelWrite<__root__>(recurse=True, writes=[ChannelWriteEntry(channel='__root__', value=<object object at 0x7be350f55f10>, skip_none=True, mapper=None)]), ChannelWrite<start:retrieve>(recurse=True, writes=[ChannelWriteEntry(channel='start:retrieve', value='__start__', skip_none=False, mapper=None)])]), 'retrieve': PregelNode(config={'tags': []}, channels=['__root__'], triggers=['start:retrieve', 'branch:decision:route:retrieve'], writers=[ChannelWrite<retrieve,__root__>(recurse=True, writes=[ChannelWriteEntry(channel='retrieve', value='retrieve', skip_none=False, mapper=None), ChannelWriteEntry(channel='__root__', value=<object object at 0x7be350f55f10>, skip_none=True, mapper=None)])]), 'generate': PregelNode(config={'tags': []}, channels=['__root__'], triggers=['retrieve', 'branch:decision:route:generate'], writers=[ChannelWrite<generate,__root__>(recurse=Tr

In [25]:
while True:
    query = input("\nAsk a question (or type 'exit'): ")

    if query.lower() == "exit":
        break

    state = create_state(query)

    # STEP 1: Retrieve
    state = retrieve_node(state)

    # STEP 2: Generate
    state = generate_node(state)

    # STEP 3: Decision
    state = decision_node(state)

    # STEP 4: HITL (if needed)
    if state["escalate"]:
        state = hitl_node(state)

    # CLEAN OUTPUT (important for demo)
    answer_lines = state["answer"].split("\n")

    print("\n💬 Answer:")
    for line in answer_lines:
        if line.strip() != "":
            print(line)


Ask a question (or type 'exit'): What is the attendance policy?



💬 Answer:
Students must maintain a minimum of 75% attendance in each subject.

Ask a question (or type 'exit'): exit
